In [ ]:
# run this notebook after 10_find_differences_siblings_python
# use this notebook to trim ibd2 segments 
# after this, run 12_analyze_trio_quad_standard_QC_R

In [ ]:
import numpy as np
import pandas 
import os

In [ ]:
bucket=os.getenv('WORKSPACE_BUCKET')

In [ ]:
f_ibd2 = bucket + "/data/ibd2_all.txt"
ibd2_all = pandas.read_table(f_ibd2)

In [ ]:
def cm_to_bp(cm, gmap_chr):
    return np.interp(cm, gmap_chr['cM'], gmap_chr['Begin'])

def bp_to_cm(bp, gmap_chr):
    return np.interp(bp, gmap_chr['Begin'], gmap_chr['cM'])

In [ ]:
ibd2_all["new_start_bp"] = None
ibd2_all["new_end_bp"] = None
ibd2_all["length_cm"] = None

In [ ]:
# Loop through chromosomes 1 to 22
for trim in [0.25, 0.5, 0.75, 1]:
    for i in range(1, 23):
        subset = ibd2_all[ibd2_all["Chr"] == i]
        if subset.empty:
            continue
        gmap_f = f'./snipar/snipar/util_data/decode_map/chr_{i}.gz'
        gmap_chr = pandas.read_table(gmap_f, sep = " ")
        start_bp = subset["start_coordinate"].tolist()
        start_cm = [bp_to_cm(x, gmap_chr) for x in start_bp]
        end_bp = subset["stop_coordinate"].tolist()
        end_cm = [bp_to_cm(x, gmap_chr) for x in end_bp]
        new_start_cm = [x+trim for x in start_cm]
        new_end_cm = [x-trim for x in end_cm]
        new_start_bp = [int(cm_to_bp(x, gmap_chr)) for x in new_start_cm]
        new_end_bp = [int(cm_to_bp(x, gmap_chr)) for x in new_end_cm]
        length_cm = [x - y for x, y in zip(new_end_cm, new_start_cm)]
        ibd2_all.loc[subset.index, "new_start_bp"] = new_start_bp
        ibd2_all.loc[subset.index, "new_end_bp"] = new_end_bp
        ibd2_all.loc[subset.index, "length_cm"] = length_cm
    
    f_out = bucket + f'/data/ibd2_all_trimmed_{trim}cM.txt'
    ibd2_all.to_csv(f_out, sep='\t', index=False)

In [ ]:
!gsutil cp $WORKSPACE_BUCKET/data/ibd2_all_trimmed_0.25cM.txt . 
!gsutil cp $WORKSPACE_BUCKET/data/ibd2_all_trimmed_0.5cM.txt . 
!gsutil cp $WORKSPACE_BUCKET/data/ibd2_all_trimmed_1cM.txt .
!gsutil cp $WORKSPACE_BUCKET/data/ibd2_all_trimmed_0.75cM.txt .